In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
from tqdm import tqdm

## Loading the data

In [ ]:
df_productcodes = pd.read_csv("/Users/pacomelefebvre/Desktop/HCSP2/MIRAGE/data/BACI_HS17_V202601/product_codes_HS17_V202601.csv")
df_contrycodes = pd.read_csv("/Users/pacomelefebvre/Desktop/HCSP2/MIRAGE/data/BACI_HS17_V202601/country_codes_V202601.csv")
df_conversion_HS17_GSEC11 = pd.read_excel(
    "/Users/pacomelefebvre/Desktop/HCSP2/MIRAGE/data/other/HS6_to_GTAP11.xlsx",
    sheet_name="H5", # I think the sheet name H5 is corresponding to HS17 since there is 
    # names from H0 to H6 with 7 versions of HS6 available online, and H0 seems older than H5
    header=1
    )
df_conversion_HS22_GSEC11 = pd.read_excel(
    "/Users/pacomelefebvre/Desktop/HCSP2/MIRAGE/data/other/HS6_to_GTAP11.xlsx",
    sheet_name="H6", # Same hypothesis, H6 must be the latest version of HS6, which is HS22
    header=1,
)

In [ ]:
df_all = pd.DataFrame()
for data in tqdm(Path("/Users/pacomelefebvre/Desktop/HCSP2/MIRAGE/data/BACI_HS17_V202601").glob("BACI_*.csv")):
    df = pd.read_csv(data)
    df_all = pd.concat([df_all, df], ignore_index=True)

df_all = df_all.rename(columns={
    "t": "Year",
    "k": "HS6",
    "i": "Exporter",
    "j": "Importer",
    "v": "k$",
    "q": "Quantity",
})

### Merging the data in one df

In [ ]:
df_merge_1 = pd.merge(
    df_all,
    df_contrycodes[["country_code", "country_iso3"]],
    left_on="Exporter",
    right_on="country_code",
    how="left",
)
df_merge_1 = (
    df_merge_1.drop(columns="Exporter")
    .rename(columns={"country_iso3": "Exporter"})
    .drop(columns="country_code")
)

In [ ]:
df_merge_2 = pd.merge(
    df_merge_1,
    df_contrycodes[["country_code", "country_iso3"]],
    left_on="Importer",
    right_on="country_code",
    how="left",
)
df_merge_2 = (
    df_merge_2.drop(columns="Importer")
    .rename(columns={"country_iso3": "Importer"})
    .drop(columns="country_code")
)

In [ ]:
df_merge_3 = pd.merge(
    df_merge_2,
    df_productcodes[["code", "description"]],
    left_on="HS6",
    right_on="code",
    how="left",
)
df_merge_3 = df_merge_3.drop(columns="HS6").rename(columns={"code": "HS6_code"})

In [ ]:
df_merge_4 = pd.merge(
    df_merge_3,
    df_conversion_HS17_GSEC11[["Code", "GSEC3_rev"]],
    left_on="HS6_code",
    right_on="Code",
    how="left",
)
df_merge_4 = df_merge_4.drop(columns="Code").rename(columns={"GSEC3_rev": "GSEC3_code"})

In [ ]:
df_fail = df_merge_4[df_merge_4["GSEC3_code"].isna()]

In [ ]:
df_fail.shape

In [ ]:
df_fail["description"].value_counts()

We can see that the mismatch only occurs for one category.

In [ ]:
df_clean = df_merge_4.groupby(["Year", "Exporter", "Importer", "GSEC3_code"]).agg({"k$": "sum", "Quantity": "sum"}).reset_index()

In [ ]:
df_clean.to_parquet(
    "/Users/pacomelefebvre/Desktop/HCSP2/MIRAGE/data/GTAP11_grouped_data.parquet", index=False
)

## Using the full data

In [ ]:
import pandas as pd

df_clean = pd.read_parquet(
    "/Users/pacomelefebvre/Desktop/HCSP2/MIRAGE/data/GTAP11_grouped_data.parquet"
)

I changed this filter for 2017, 2022, and 2024.

In [ ]:
df_clean = df_clean[df_clean["Year"] == 2024]

In [ ]:
EU27 = [
    "AUT",
    "BEL",
    "BGR",
    "HRV",
    "CYP",
    "CZE",
    "DNK",
    "EST",
    "FIN",
    "FRA",
    "DEU",
    "GRC",
    "HUN",
    "IRL",
    "ITA",
    "LVA",
    "LTU",
    "LUX",
    "MLT",
    "NLD",
    "POL",
    "PRT",
    "ROU",
    "SVK",
    "SVN",
    "ESP",
    "SWE",
]

eu_map = {code: "UE27" for code in EU27}

df_UE = df_clean.copy().replace({"Exporter": eu_map, "Importer": eu_map})


In [ ]:
df_UE_clean = df_UE[(df_UE["Exporter"] == "UE27") & (df_UE["Importer"] == "CHN") | (df_UE["Exporter"] == "CHN") & (df_UE["Importer"] == "UE27")]

In [ ]:
exports_ue_chine = (
    df_UE_clean[(df_UE_clean["Exporter"] == "UE27") & (df_UE_clean["Importer"] == "CHN")]
    .groupby("GSEC3_code")["k$"]
    .sum()
)
imports_ue_chine = (
    df_UE_clean[(df_UE_clean["Exporter"] == "CHN") & (df_UE_clean["Importer"] == "UE27")]
    .groupby("GSEC3_code")["k$"]
    .sum()
)
df_bilan = pd.DataFrame(
    {
        "Exports_UE_vers_Chine": exports_ue_chine,
        "Imports_UE_depuis_Chine": imports_ue_chine,
    }
).fillna(0)

df_bilan["Solde_commercial"] = (
    df_bilan["Exports_UE_vers_Chine"] - df_bilan["Imports_UE_depuis_Chine"]
)
df_bilan = df_bilan.reset_index()


In [ ]:
df_bilan.to_csv(
    "/Users/pacomelefebvre/Desktop/HCSP2/MIRAGE/data/bilan_commercial_UE_CHN_2024.csv", index=False
)

## Chinese market share

In [5]:
import pandas as pd

df_clean = pd.read_parquet(
    "/Users/pacomelefebvre/Desktop/HCSP2/MIRAGE/data/GTAP11_grouped_data.parquet"
)


In [6]:
df_export_per_cat = df_clean.groupby(by="GSEC3_code")["k$"].sum().rename("k$_tot")

In [7]:
df_export_per_cat_CHN = (
    df_clean[df_clean["Exporter"] == "CHN"]
    .groupby(by="GSEC3_code")["k$"]
    .sum()
    .rename("k$_CHN")
)

In [8]:
df_market_share = pd.merge(
    left=df_export_per_cat, right=df_export_per_cat_CHN, on="GSEC3_code", how="left"
)
df_market_share["CHN_market_share (%)"] = (
    100 * df_market_share["k$_CHN"] / df_market_share["k$_tot"]
)
df_market_share.to_csv(
    "/Users/pacomelefebvre/Desktop/HCSP2/MIRAGE/data/China_market_share.csv"
)